In [ ]:
using DelimitedFiles;       #Paquetería para leer y escribir datos
using CairoMakie;           #Paquetería para graficar datos
using FFTW;                 #Paquetería para realizar análisis de Fourier
using LaTeXStrings;         #Paquetería para emplear texto en LaTeX en las gráficas
using LsqFit;               #Paquetería para realizar un ajuste por mínimos cuadrados
using Statistics;           #Paquetería para realizar cálculos de estadística básica

#Definimos la función que calcula nuestro aproximante estadístico
AproxLambda(NSides) = (2π / (1 - cos(2π / NSides)));

DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/SI_Fig5-A_SigmaSquare_2D/"; #Ruta de acceso a los datos para analizar

In [ ]:
#Diccionario con los valores de la densidad numérica de los sistemas cuasiperiódicos en 2D
Rho_Dict = Dict(
                5  => 1.2328979808609704,
                7  => 1.2517957581185175,
                9  => 1.260284085456272,
                11 => 1.2645739922427748,
                13 => 1.2670366156186388,
                15 => 1.2685820147323164,
                17 => 1.2696141740269873,
                19 => 1.2703373895977548,
                21 => 1.2708642307351703,
                23 => 1.2712590235307708,
                25 => 1.271563821555834,
                27 => 1.271802352106815,
                29 => 1.2719940413420914,
                31 => 1.2721488757491926
               );

### Gráfica de hiperuniformidad con cálculo de Λ_∞ en CairoMakie

In [ ]:
##########################################################################################################################################################################
#                                                                   Datos del sistema cuasiperiódico
##########################################################################################################################################################################
NSides = 31;                #Simetría rotacional de los sistemas cuasperiódicos a analizar
Radio = 500;                #Radio de las vecindades circulares
Pasos = 1e5;                #Número de pasos a dar desde R = 0 hasta R = Radio para variar el radio de la ventana circular de N(R)
ΔStep = Radio/Pasos;        #Salto en los valores de R
##########################################################################################################################################################################
#                                                                  Lectura de los datos de la Sigma^2
##########################################################################################################################################################################
σ2 = vec(readdlm(DataPath * "Torquato_NR_N$(NSides)_Alfa0P0_R$(Radio)_Step1e5_SigmaCuadrada.csv"));
##########################################################################################################################################################################
#                                                  Cálculo de la densidad promedio y factor de normalización (Torquato)
##########################################################################################################################################################################
Rho = Rho_Dict[NSides];     #Densidad numérica de sitios en la retícula cuasiperiódica
FN = 2*sqrt(π*Rho);         #Factor de normalización para mantener los resultados independientes de la densidad de puntos
##########################################################################################################################################################################
#                                                           Cálculo del arreglo de radios para la sigma^2
##########################################################################################################################################################################
R = ΔStep:ΔStep:Radio;      #Intervalo de radios analizados para calcular la función σ^2(R)
R = FN .* R;                #Normalización de los radios por el Factor de Torquato
##########################################################################################################################################################################
#                                                               Cálculo de la longitud de escala λ
##########################################################################################################################################################################
λ = AproxLambda(NSides);    #Longitud de escala de nuestro sistema
##########################################################################################################################################################################
#                                                                   Cálculo de la Λ_∞
##########################################################################################################################################################################
Start_λ = findfirst(x -> x >= λ, R);                                        #Indice del primer radio igual o mayor a la escala de longitud λ
End_λ = findfirst(x -> x >= Int(floor(R[end]/λ))*λ, R);                     #Indice del mayor radio que cierra un ciclo λ

Sigma2_R_Acotado = (σ2 ./ R)[Start_λ:End_λ];                                #Datos de σ^2(R) / R correspondiente únicamente a los ciclos seleccionados
Media = mean(Sigma2_R_Acotado);                                             #Promedio de los σ^2(R) / R
STD_Media = std(Sigma2_R_Acotado);                                          #Desviación estándar de los datos
println("El valor promedio de los datos es: $(Media)")
println("La desviación estándar con respecto al promedio es: $(STD_Media)")
##########################################################################################################################################################################
#                                                                Gráfica de la σ^2(R) con la Λ_∞
##########################################################################################################################################################################
# --- Definición de las características del lienzo y las subgráficas en él ---
Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
Sigma2_Ax = Axis(
                 Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                 title = L"N = %$(NSides)",                                   #Título de la gráfica
                 xlabel = L"R",                                               #Etiqueta que aparece en el eje horizontal
                 ylabel = L"\sigma^{2}(R)/R",                                 #Etiqueta que aparece en el eje vertical
                 titlesize = 55,                                              #Tamaño del título
                 xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                 ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                 xticklabelsize = 40,                                         #Tamaño para el eje X
                 yticklabelsize = 40,                                         #Tamaño para el eje Y
                 xticksize = 25,                                              #Tamaño de los ticks horizontales
                 yticksize = 25,                                              #Tamaño de los ticks verticales
                 limits = ((0, nothing), nothing),                            #Límites de la visualización para la gráfica
                 #xscale = log10,
                 #yscale = log10
                )
hidespines!(Sigma2_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
hidedecorations!(
                 Sigma2_Ax,
                 label = false,           #Se oculta o no las etiquetas a los ejes
                 ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                 ticks = false            #Se oculta o no los ticks de los ejes
                )
# --- Gráfica de los datos de la σ^2(R) ---
Final = 0; #Datos del final a eliminar (por errores en inconsistencias de tamaño en vecindades)
lines!(Sigma2_Ax, R[1:(end - Final)], (σ2 ./ R)[1:(end - Final)])
# --- Líneas verticales con el inicio y final del cálculo de Λ_∞
vlines!(
        Sigma2_Ax, [R[Start_λ], R[End_λ]],
        linestyle = :dash,
        alpha = 1,
        linewidth = 8,
        color = :black,
        label = L"\lambda = \frac{2 \pi}{1 - \cos \left( \frac{2 \pi}{%$(NSides)} \right)} \approx %$(round(λ, digits = 2))",
       )
# --- Línea horizontal con el valor de Λ_∞
lines!(
       Sigma2_Ax, [R[Start_λ], R[End_λ]], [Media, Media],
       alpha = 1,
       linewidth = 8,
       color = :red
      )
###################################################################################################################
#                                            Guardamos las gráficas    
###################################################################################################################
Fig

### Gráfica de $\Lambda_{\infty}$ como función de N con barras de error

In [ ]:
#Modelo: Promedio
#Starting Point = λ
Dict_Lambda_Infty_STD = Dict(
                             5  => [0.14842677130532758, 0.02693208062674656],
                             7  => [0.20599942249634387, 0.02889749910608079],
                             9  => [0.20511073236241031, 0.025078052118710375],
                             11 => [0.28812778652572674, 0.05343895152132179],
                             13 => [0.3478715456767856, 0.07341374061357663],
                             15 => [0.34865315655359685, 0.08051135345795574],
                             17 => [0.4593732477904627, 0.11864843159480597],
                             19 => [0.5337295342446887, 0.145608944161264],
                             21 => [0.5535341355465558, 0.15582047558893325],
                             23 => [0.6813762592148858, 0.1963541228475375],
                             25 => [0.7593901994594552, 0.22370865721408364],
                             27 => [0.7848784960127629, 0.23360456762197016],
                             29 => [0.9205104301894491, 0.27222890717914594],
                             31 => [1.0268510348123494, 0.3089886592548087]
                            );

# --- Definimos el lienzo general y el eje donde se graficará la primera imagen en el lienzo completo ---
Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
LambdaInfty_Ax = Axis(
                      Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                      xlabel = L"N",                                               #Etiqueta que aparece en el eje horizontal
                      ylabel = L"\Lambda_{\infty}",                                #Etiqueta que aparece en el eje vertical
                      titlesize = 55,                                              #Tamaño del título
                      xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                      ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                      xticklabelsize = 40,                                         #Tamaño para el eje X
                      yticklabelsize = 40,                                         #Tamaño para el eje Y
                      xticksize = 25,                                              #Tamaño de los ticks horizontales
                      yticksize = 25,                                              #Tamaño de los ticks verticales
                      limits = (nothing, nothing),                                 #Límites de la visualización para la gráfica
                     );
hidespines!(LambdaInfty_Ax, :t, :r);      #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
hidedecorations!(
                 LambdaInfty_Ax,
                 label = false,           #Se oculta o no las etiquetas a los ejes
                 ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                 ticks = false            #Se oculta o no los ticks de los ejes
                );

N_Array = 5:2:31;
Lambdas_Array = [Dict_Lambda_Infty_STD[NSides][1] for NSides in N_Array];
STD_Array = [Dict_Lambda_Infty_STD[NSides][2] for NSides in N_Array];

@. model(x, p) = p[1] + p[2] * (x^p[3])
p0 = [0.5, 0.5, 0.5]
fit = curve_fit(model, N_Array, Lambdas_Array, p0)
K = coef(fit)
println(K)

lines!(
       LambdaInfty_Ax, N_Array, K[1] .+ (K[2] .* (N_Array .^ K[3])),
       label = L"\Lambda_{\infty} = %$(round(K[1], digits = 3)) + %$(round(K[2], digits = 3)) N^{%$(round(K[3], digits = 3))}"
      );
CairoMakie.scatter!(LambdaInfty_Ax, N_Array, Lambdas_Array)
errorbars!(
           LambdaInfty_Ax, N_Array, Lambdas_Array, STD_Array, 
           whiskerwidth = 10,  #Ancho de la línea horizontal superior/inferior
           color = :black,
           linewidth = 1.5
          )

Fig